<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/C%C3%B3digo_para_o_Exp_3A_SistControle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from numpy.polynomial import Polynomial

def polinomioCaracteristico(A):
  A = -A; s = Polynomial([0, 1])
  if(A.shape[0] == 3):
    plus = (s+A[0][0])*(s+A[1][1])*(s+A[2][2]) + (A[0][1]*A[1][2]*A[2][0]) + (A[0][2]*A[1][0]*A[2][1])
    minus = (A[2][0]*(s+A[1][1])*A[0][2]) + (A[2][1]*A[1][2]*(s+A[0][0])) + ((s+A[2][2])*A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 2):
    plus = (s+A[0][0])*(s+A[1][1])
    minus = (A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 1):
    poly = s+A[0][0]; polos = poly.roots()
  else:
    print("Não é possível calcular o polinômio característico.")
  polos = np.round(polos, 3)
  print("Polinômio característico: " + str(poly))
  print("Polos: " + str(polos))
  for i in range(len(polos)):
    if(polos[i].real > 0):
      print("O sistema é instável.")
      break
  print("O sistema é estável.")

def matrizControlabilidade(A, B):
  n = A.shape[0]
  U = np.zeros((n,n))
  for i in range(n):
    U[:,i:i+1] = (np.linalg.matrix_power(A,i))@B
  posto = np.linalg.matrix_rank(U)
  return U, posto

def matrizObservabilidade(A, C):
  n = A.shape[0]
  V = np.zeros((n,n))
  for i in range(n):
    V[i:i+1, :] = C@(np.linalg.matrix_power(A,i))
  posto = np.linalg.matrix_rank(V)
  return V, posto

def realimentaçãoDeEstado(A, U, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; qc = np.zeros((n,n))
  for i in range(n+1):
    qc += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invU = np.linalg.inv(U)
  linha = np.zeros((1,n)); linha[0][-1] = 1
  return -linha@(invU@qc)

def observadorDeEstado(A, V, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; ql = np.zeros((n,n))
  for i in range(n+1):
    ql += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invV = np.linalg.inv(V)
  coluna = np.zeros((n,1)); coluna[-1][0] = 1
  return ql@invV@coluna

In [3]:
g=981
A1 = 15.5179
A2 = A1
a1=0.17813919765
a2=a1
L20 = 15
L10=((a2/a1)**2)*L20
Km=3.7845
alpha1=-(a1/A1)*np.sqrt(g/(2*L10))
alpha3=(a1/A2)*np.sqrt(g/(2*L10))
alpha4=-(a2/A2)*np.sqrt(g/(2*L20))
beta1=Km/A1

A = np.array([[alpha1, 0],
              [alpha3, alpha4]])
B = np.array([[beta1],
              [0]])
C = np.array([[0, 1]])

polinomioCaracteristico(A); print("\nA = " + str(A))

(U, posto) = matrizControlabilidade(A, B); print("U = " + str(U))
if(posto == A.shape[0]):
  print("Controlável")
else:
  print("Não é controlável")

(V, posto) = matrizObservabilidade(A, C); print("\nV = " + str(V))
if(posto == A.shape[0]):
  print("Observável")
else:
  print("Não é observável")

s = np.array([-4, -4]); K = realimentaçãoDeEstado(A, U, s); print("\nK = " + str(K))
s = np.array([-3, -3]); L = observadorDeEstado(A, V, s); print("\nL = " + str(L))

Polinômio característico: 0.00430924 + 0.13128963·x + 1.0·x²
Polos: [-0.066 -0.066]
O sistema é estável.

A = [[-0.06564481  0.        ]
 [ 0.06564481 -0.06564481]]
U = [[ 0.24387965 -0.01600943]
 [ 0.          0.01600943]]
Controlável

V = [[ 0.          1.        ]
 [ 0.06564481 -0.06564481]]
Observável

K = [[ -32.26472736 -966.87682166]]

L = [[131.16710811]
 [  5.86871037]]
